# Loading and preparing data 

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
from tcrdist.repertoire import TCRrep
from tcrdist.join import join_by_dist

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tcrdist.sparse import add_sparse_pwd

import pandas as pd
import numpy as np

from collections import Counter

import networkx as nx
import umap

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches # Legend

import os
import sys
import warnings

## Root directory

In [ ]:
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

# Parameters 

In [ ]:
# Set 
radius_dist_thr = 36

In [ ]:
# Set information columns 
clone_cols = ['v_a_gene','j_a_gene','cdr3_a_aa', 'v_b_gene','j_b_gene','cdr3_b_aa']
meta_cols = ['barcode', 'patient_id', 'time_point', 'hto_demux_class', 'genotype_class']
meta_iedb_cols = ['epitope_organism_iedb', 'epitope_molecule_iedb', 'epitope_iri_iedb']
meta_vdjdb_cols = ['antigen_species_vdjdb', 'antigen_gene_vdjdb', 'antigen_epitope_vdjdb']

In [ ]:
# Color mapping 
color_map = {
    
    'Blood': '#6E0B53',
    'Skin': '#006E6E', 

    'Baseline': '#74ABAB',
    'Tx': '#6E5E0B',
    'D14': '#BA9E09',
    'D100': '#006E6E',
    'GVHD': '#B3254D', 

    'Patient_1': '#003B3B',
    'Patient_2': '#006E6E',
    'Patient_3': '#43E0E0',
    'Patient_4': '#00DEA6',
    'Patient_5': '#3AC2A0',
    'Patient_6': '#00A87E',
    'Patient_7': '#007558',
    'Patient_8': '#CE677D',
    'Patient_9': '#C42749',
    'Patient_10': '#911D36',
    'Patient_11': '#793644',
    'Patient_12': '#912154', 

    'Host': '#00B0AD', 
    'Donor': '#C21398'

}

# Import cell df 

In [ ]:
# Load cell data frame
cell_df = pd.read_csv('result/ir/tcrdist/cell_df.csv', index_col=0)

# Graph based meta clone 

## Prepare graph clone df

In [ ]:
graph_clone_df = cell_df.copy()

In [ ]:
# Aggregate on unique TCRs
graph_clone_df = graph_clone_df.dropna(subset=clone_cols).groupby(clone_cols, dropna=False, sort=False).size().reset_index(name='count')

In [ ]:
# Compute TCRdist distance
tr = TCRrep(cell_df=graph_clone_df, organism='human', chains=['alpha', 'beta'], db_file='alphabeta_gammadelta_db.tsv', compute_distances=True, deduplicate=False) # Manually dedublicated - need to take extreme care wheather clone and cell df are in same order. 

In [ ]:
# Compute combined alpha beta distance 
pw = tr.pw_alpha + tr.pw_beta

In [ ]:
# Get clean clone df
graph_clone_df = tr.clone_df.reset_index(drop=True)

## Perform grid serach on list of radii 

In [ ]:
# Perform grid search (gs) on a list of radii
radius_gs = list(range(1, 51))
graph_clone_min_size = 2

gs_res = []

for radius in radius_gs:
    
    adj = csr_matrix(pw <= radius)
    adj.setdiag(0)
    adj.eliminate_zeros()

    graph_clone_total, graph_clone_id = connected_components(adj, directed=False)

    graph_clone_n = pd.Series(graph_clone_id).value_counts().sort_values(ascending=False)
    meta_sizes = graph_clone_n[graph_clone_n >= graph_clone_min_size]

    gs_res.append(
        
        {
            
            'radius': radius,
            'n_graph_clones': int(meta_sizes.shape[0]),
            'min_graph_clone_size': int(meta_sizes.min()) if len(meta_sizes) else 0,
            'median_graph_clone_size': float(meta_sizes.median()) if len(meta_sizes) else np.nan,
            'max_graph_clone_size': int(meta_sizes.max()) if len(meta_sizes) else 0
        
        }
                   )

gs_df = pd.DataFrame(gs_res)

In [ ]:
# Plot grid search results  
plot_cols = [
    
    'n_graph_clones',
    'max_graph_clone_size',
    
]

fig, axes = plt.subplots(1, 2, figsize=(5, 5), sharex=True)
axes = axes.ravel()

for ax, col in zip(axes, plot_cols):
    ax.plot(gs_df['radius'], gs_df[col], marker='o', linewidth=1.5, markersize=4)
    ax.set_title(col.replace('_', ' ').capitalize())
    ax.set_xlabel('Radius')
    ax.set_ylabel(col.replace('_', ' ').capitalize())
    ax.grid(True, alpha=0.3)

fig.suptitle('Graph-clone summary across TCRdist radius thresholds', fontsize=14)
fig.tight_layout()
plt.show()

## Graph clone ID

In [ ]:
# same adjacency definition as in grid search
adj = csr_matrix(((pw >= 0) & (pw <= radius_dist_thr)))
adj.setdiag(False)
adj.eliminate_zeros()

In [ ]:
# connected component
graph_clone_total, graph_clone_id = connected_components(adj, directed=False)

In [ ]:
# Numeric meta clone ID
graph_clone_df['graph_clone_id_int'] = graph_clone_id

# Meta clone size
graph_clone_df['graph_clone_n'] = graph_clone_df.groupby('graph_clone_id_int')['graph_clone_id_int'].transform('size')

# Rank clones by size and make ID labels 
graph_clone_id = graph_clone_df[['graph_clone_id_int', 'graph_clone_n']].drop_duplicates().sort_values(['graph_clone_n', 'graph_clone_id_int'], ascending=[False, True]).reset_index(drop=True)
graph_clone_id['graph_clone_rank'] = graph_clone_id.index
graph_clone_id['graph_clone_id'] = 'graph_clone_' + graph_clone_id['graph_clone_rank'].astype(str)

# Map back to graph_clone_df
graph_clone_df = graph_clone_df.merge(graph_clone_id[['graph_clone_id', 'graph_clone_id_int']], on='graph_clone_id_int', how='left').drop(columns='graph_clone_id_int')

In [ ]:
# Store clone df
# graph_clone_df.to_csv('result/ir/tcrdist/graph_clone_df.csv')

# Graph visualization (Graph Clone)

In [235]:
graph_vis_df = cell_df.copy()

In [236]:
# Make graphe visualization df on meta columns
graph_vis_df['subject'] = graph_vis_df[['patient_id','time_point','hto_demux_class', 'genotype_class']].astype(str).agg('_'.join, axis=1)
graph_vis_df = graph_vis_df.dropna(subset=clone_cols+['subject', 'patient_id','time_point','hto_demux_class', 'genotype_class']).groupby(clone_cols+['subject', 'patient_id','time_point','hto_demux_class', 'genotype_class'], dropna=False, sort=False).size().reset_index(name='count')

In [237]:
# Add graph clone id
graph_vis_df = graph_vis_df.merge(graph_clone_df[clone_cols+['graph_clone_id']], on=clone_cols, how='left')

In [238]:
# Count graphe clone sharing accross groups
graph_vis_df['graph_clone_n'] = graph_vis_df.groupby('graph_clone_id')['graph_clone_id'].transform('size')

In [239]:
# Compute TCRdist distance
tr = TCRrep(cell_df=graph_vis_df, organism='human', chains=['alpha', 'beta'], db_file='alphabeta_gammadelta_db.tsv', compute_distances=True, deduplicate=False) # Manually dedublicated - need to take extreme care wheather clone and cell df are in same order. 

In [240]:
# Compute combined alpha beta distance 
pw = tr.pw_alpha + tr.pw_beta

In [241]:
# Get clean clone df
graph_vis_df = tr.clone_df.reset_index(drop=True)

## Atlas collection 

In [216]:
clone_mask = graph_vis_df["graph_clone_n"] >= 2

pw_atlas = pw[clone_mask][:, clone_mask]
cell_atlas_df = graph_vis_df.loc[clone_mask].copy().reset_index(drop=True)

In [217]:
edge_mask = (pw_atlas >= 0) & (pw_atlas <= radius_dist_thr)
np.fill_diagonal(edge_mask, False)

In [218]:
G = nx.from_numpy_array(edge_mask)

In [219]:
# Set edge weight
for i, j in G.edges():
    
    G[i][j]["weight"] = 1 - (pw_atlas[i, j] / radius_dist_thr)

# Node size
node_sizes = np.sqrt(cell_atlas_df['count']) * 3

# Node color
node_color = cell_atlas_df['patient_id'].map(color_map).to_numpy()

# Graph positions 
pos = nx.nx_agraph.graphviz_layout(G, prog="neato")

In [220]:
G_k = list(nx.connected_components(G))

In [221]:
plt.figure(figsize=(2, 2))

G_ij = list(nx.connected_components(G))

for i, j in enumerate(G_ij):
    
    sg = G.subgraph(j)

    nx.draw_networkx_edges(
        sg,
        pos,
        alpha=1.0,
        width=0.5
    )

    nx.draw_networkx_nodes(
        
        sg,
        pos,
        nodelist=list(sg.nodes()),
        node_size=node_sizes[list(sg.nodes())],
        node_color=node_color[list(sg.nodes())],
        linewidths=0.2,
        edgecolors="black",
        alpha=1.0
        
    )

plt.axis("off")
plt.tight_layout()

plt.savefig("result/figures/figure_5/graph_clone_atlas.pdf", format="pdf", bbox_inches="tight")

plt.close()

## UMAP

In [ ]:
# Filter data for meta clones with at elast 2 clones
clone_mask = graph_vis_df['graph_clone_n'] >= 2

pw_umap = pw[clone_mask][:, clone_mask]
cell_umap_df = graph_vis_df.loc[clone_mask].copy()

In [ ]:
# Compute umap 
embedding = umap.UMAP(
    
    metric='precomputed', 
    n_neighbors=30,
    min_dist=0.3,
    random_state=0

).fit_transform(pw_umap)

In [ ]:
cell_umap_df['UMAP_1'] = embedding[:, 0]
cell_umap_df['UMAP_2'] = embedding[:, 1]

In [ ]:
color = 'time_point'
cats = pd.Categorical(cell_umap_df[color])
cell_umap_df['color_hex'] = cell_umap_df[color].map(color_map)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(
    
    cell_umap_df['UMAP_1'],
    cell_umap_df['UMAP_2'],
    c=cell_umap_df['color_hex'],
    s=np.sqrt(cell_umap_df['count']) * 10,
    edgecolors='black',
    linewidths=0.2
    
)

# create legend
legend_df = cell_umap_df[[color, 'color_hex']].drop_duplicates()
handles = [mpatches.Patch(color=row['color_hex'], label=row[color]) for _, row in legend_df.iterrows()]
ax.legend(handles=handles, title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xlabel('UMAP1')
ax.set_ylabel('UMAP2')
ax.set_aspect('equal')

plt.show()

# Graph visualization (Clone)

In [197]:
graph_vis_df = cell_df.copy()

In [198]:
# Make graphe visualization df on meta columns
graph_vis_df['subject'] = graph_vis_df[['patient_id','time_point','hto_demux_class', 'genotype_class', 'clone_id']].astype(str).agg('_'.join, axis=1)
graph_vis_df = graph_vis_df.dropna(subset=clone_cols+['subject', 'patient_id','time_point','hto_demux_class', 'genotype_class', 'clone_id']).groupby(clone_cols+['subject', 'patient_id','time_point','hto_demux_class', 'genotype_class', 'clone_id'], dropna=False, sort=False).size().reset_index(name='count')

In [199]:
# Count graphe clone sharing accross groups
graph_vis_df['clone_n'] = graph_vis_df.groupby('clone_id')['clone_id'].transform('size')

In [200]:
# Compute TCRdist distance
tr = TCRrep(cell_df=graph_vis_df, organism='human', chains=['alpha', 'beta'], db_file='alphabeta_gammadelta_db.tsv', compute_distances=True, deduplicate=False) # Manually dedublicated - need to take extreme care wheather clone and cell df are in same order. 

In [201]:
# Compute combined alpha beta distance 
pw = tr.pw_alpha + tr.pw_beta

In [202]:
# Get clean clone df
graph_vis_df = tr.clone_df.reset_index(drop=True)

## Atlas collection 

In [203]:
clone_mask = graph_vis_df["clone_n"] >= 2

pw_atlas = pw[clone_mask][:, clone_mask]
cell_atlas_df = graph_vis_df.loc[clone_mask].copy().reset_index(drop=True)

In [204]:
edge_mask = (pw_atlas >= 0) & (pw_atlas <= radius_dist_thr)
np.fill_diagonal(edge_mask, False)

In [205]:
G = nx.from_numpy_array(edge_mask)

In [206]:
# Set edge weight
for i, j in G.edges():
    
    G[i][j]["weight"] = 1 - (pw_atlas[i, j] / radius_dist_thr)

# Node size
node_sizes = np.sqrt(cell_atlas_df['count']) * 3

# Node color
node_color = cell_atlas_df['patient_id'].map(color_map).to_numpy()

# Graph positions 
pos = nx.nx_agraph.graphviz_layout(G, prog="neato")

In [207]:
G_k = list(nx.connected_components(G))

In [208]:
plt.figure(figsize=(2, 2))

G_ij = list(nx.connected_components(G))

for i, j in enumerate(G_ij):
    
    sg = G.subgraph(j)

    nx.draw_networkx_edges(
        sg,
        pos,
        alpha=1.0,
        width=0.5
    )

    nx.draw_networkx_nodes(
        
        sg,
        pos,
        nodelist=list(sg.nodes()),
        node_size=node_sizes[list(sg.nodes())],
        node_color=node_color[list(sg.nodes())],
        linewidths=0.2,
        edgecolors="black",
        alpha=1.0
        
    )

plt.axis("off")
plt.tight_layout()

plt.savefig("result/figures/figure_5/clone_atlas.pdf", format="pdf", bbox_inches="tight")

plt.close()

## UMAP

In [ ]:
# Filter data for meta clones with at elast 2 clones
clone_mask = graph_vis_df['clone_n'] >= 2

pw_umap = pw[clone_mask][:, clone_mask]
cell_umap_df = graph_vis_df.loc[clone_mask].copy()

In [ ]:
# Compute umap 
embedding = umap.UMAP(
    
    metric='precomputed', 
    n_neighbors=30,
    min_dist=0.3,
    random_state=0

).fit_transform(pw_umap)

In [ ]:
cell_umap_df['UMAP_1'] = embedding[:, 0]
cell_umap_df['UMAP_2'] = embedding[:, 1]

In [ ]:
color = 'time_point'
cats = pd.Categorical(cell_umap_df[color])
cell_umap_df['color_hex'] = cell_umap_df[color].map(color_map)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(
    
    cell_umap_df['UMAP_1'],
    cell_umap_df['UMAP_2'],
    c=cell_umap_df['color_hex'],
    s=np.sqrt(cell_umap_df['count']) * 10,
    edgecolors='black',
    linewidths=0.2
    
)

# create legend
legend_df = cell_umap_df[[color, 'color_hex']].drop_duplicates()
handles = [mpatches.Patch(color=row['color_hex'], label=row[color]) for _, row in legend_df.iterrows()]
ax.legend(handles=handles, title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xlabel('UMAP1')
ax.set_ylabel('UMAP2')
ax.set_aspect('equal')

plt.show()

# Annotate TCR with IEDB

In [ ]:
cell_iedb_df = pd.read_csv('result/ir/tcrdist/cell_iedb_df.csv', index_col=0)

In [ ]:
# Make clone id to map meta data with clones
cell_iedb_df['clone'] = 'clone_' + cell_iedb_df.groupby(clone_cols, dropna=False).ngroup().astype('int64').astype(str)

In [ ]:
# Subset to clone IEDB data frame
clone_iedb_df = cell_iedb_df[['clone'] + clone_cols].drop_duplicates().reset_index(drop=True).assign(count=1)

In [ ]:
tr_qry = TCRrep(
    
    cell_df=graph_clone_df,
    organism='human',
    chains=['alpha','beta'],
    compute_distances=False, 
    deduplicate=True
    
)

tr_ref = TCRrep(
    
    cell_df=clone_iedb_df,
    organism='human',
    chains=['alpha','beta'],
    compute_distances=False, 
    deduplicate=True
    
)

In [ ]:
clone_qry_df = tr_qry.clone_df
clone_ref_df = tr_ref.clone_df

In [ ]:
# Copute distance between qry and ref clones
tr_qry.compute_sparse_rect_distances(
    
    df=clone_qry_df,
    df2=clone_ref_df,
    radius=radius_dist_thr
    
)

# Store combined ab distance
tr_qry.rw_alpha_beta = add_sparse_pwd(tr_qry.rw_beta, tr_qry.rw_alpha)

In [ ]:
# Get hits between qry and ref
clone_iedb_map = join_by_dist(
    
    how='left',
    csrmat=tr_qry.rw_alpha_beta,
    left_df=clone_qry_df,
    right_df=clone_ref_df,
    left_cols=clone_qry_df.columns.to_list(),
    right_cols=clone_ref_df.columns.to_list(),
    left_suffix='',
    right_suffix='_iedb',
    max_n=clone_ref_df.shape[0],
    radius=radius_dist_thr
    
)

In [ ]:
# Get meta iedb for cones that mapped 
meta_iedb = cell_iedb_df[cell_iedb_df.clone.isin(clone_iedb_map.clone_iedb)][meta_iedb_cols+['clone']].rename(columns={'clone':'clone_iedb'})

In [ ]:
# Annotated mapping 
clone_iedb_map = clone_iedb_map.merge(meta_iedb, on='clone_iedb', how='left')

In [ ]:
# Save results
clone_iedb_map.to_csv('result/ir/tcrdist/clone_iedb_map.csv')

# Annotate TCR with VDJDB

In [ ]:
cell_vdjdb_df = pd.read_csv('result/ir/tcrdist/cell_vdjdb_df.csv', index_col=0)

In [ ]:
# Make clone id to map meta data with clones
cell_vdjdb_df['clone'] = 'clone_' + cell_vdjdb_df.groupby(clone_cols, dropna=False).ngroup().astype('int64').astype(str)

In [ ]:
# Subset to clone VDJDB data frame
clone_vdjdb_df = cell_vdjdb_df[['clone'] + clone_cols].drop_duplicates().reset_index(drop=True).assign(count=1)

In [ ]:
tr_qry = TCRrep(
    
    cell_df=graph_clone_df,
    organism='human',
    chains=['alpha','beta'],
    compute_distances=False, 
    deduplicate=True
    
)

tr_ref = TCRrep(
    
    cell_df=clone_vdjdb_df,
    organism='human',
    chains=['alpha','beta'],
    compute_distances=False, 
    deduplicate=True
    
)

In [ ]:
clone_qry_df = tr_qry.clone_df
clone_ref_df = tr_ref.clone_df

In [ ]:
# Copute distance between qry and ref clones
tr_qry.compute_sparse_rect_distances(
    
    df=clone_qry_df,
    df2=clone_ref_df,
    radius=radius_dist_thr
    
)

In [ ]:
# Store combined ab distance
tr_qry.rw_alpha_beta = add_sparse_pwd(tr_qry.rw_beta, tr_qry.rw_alpha)

In [ ]:
# Get hits between qry and ref
clone_vdjdb_map = join_by_dist(
    
    how='left',
    csrmat=tr_qry.rw_alpha_beta,
    left_df=clone_qry_df,
    right_df=clone_ref_df,
    left_cols=clone_qry_df.columns.to_list(),
    right_cols=clone_ref_df.columns.to_list(),
    left_suffix='',
    right_suffix='_vdjdb',
    max_n=clone_ref_df.shape[0],
    radius=radius_dist_thr
    
)

In [ ]:
# Get meta vdjdb for cones that mapped 
meta_vdjdb = cell_vdjdb_df[cell_vdjdb_df.clone.isin(clone_vdjdb_map.clone_vdjdb)][meta_vdjdb_cols+['clone']].rename(columns={'clone':'clone_vdjdb'})

In [ ]:
# Annotated mapping 
clone_vdjdb_map = clone_vdjdb_map.merge(meta_vdjdb, on='clone_vdjdb', how='left')

In [ ]:
# Save results
clone_vdjdb_map.to_csv('result/ir/tcrdist/clone_vdjdb_map.csv')